# Huấn luyện mô hình quyết định kiểu Jev (NT532)

Notebook train hai mô hình song song rồi chấm trên test và test_shift:

- `jev1`: EmbeddingGemma 2 + LoRA + đầu Choice/Boolean, chạy trên GPU 0.
- `heads_only`: đối chứng, đóng băng backbone, chỉ train đầu, chạy trên GPU 1.

**Yêu cầu:** Accelerator GPU T4 x2, Internet ON, đã thêm Dataset chứa 4 file `train/val/test/test_shift.jsonl`
(xem `kaggle/README.md`). Chạy được ở chế độ Save & Run All (không cần tương tác).

**Kết quả:** `/kaggle/working/jev_results.zip` gồm `runs/decider/jev/jev1` và `heads_only`
(adapter LoRA, đầu, config, eval.json, log), không gồm mô hình gốc.

**Lưu ý:** model `google/embeddinggemma-2` có thể cần chấp nhận điều khoản trên Hugging Face; nếu tải lỗi 401/403
hãy thêm HF_TOKEN vào Kaggle Secrets.

In [ ]:
!nvidia-smi
!git clone https://github.com/knight3007/Project-NT532.R12.git /kaggle/working/repo

## Cài thư viện
Dùng torch có sẵn của Kaggle (không cài lại). Chỉ thêm sentence-transformers, transformers>=5, peft và
cài gói `nt532` ở chế độ editable (không kéo thêm phụ thuộc). Phần quyết định chỉ cần numpy và pyyaml.

In [ ]:
!pip install -q "sentence-transformers>=5" "transformers>=5" "peft>=0.13" pyyaml psutil
!pip install -q -e /kaggle/working/repo/pi --no-deps
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.device_count())
assert int(transformers.__version__.split(".")[0]) >= 5, "cần transformers>=5"
print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported(), "(T4 không có bf16 gốc nên dùng fp32)")

## Gắn dữ liệu
Tìm `train.jsonl` bất kỳ đâu dưới `/kaggle/input` rồi chép 4 file vào `<repo>/runs/decider/scenarios/`
(đường dẫn mà script mong đợi, vì `REPO_ROOT` là thư mục repo).

In [ ]:
import glob, os, shutil
hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert hits, "không thấy train.jsonl dưới /kaggle/input; hãy Add Data với dataset nt532-scenarios"
src = os.path.dirname(hits[0])
dst = "/kaggle/working/repo/runs/decider/scenarios"
os.makedirs(dst, exist_ok=True)
for n in ("train", "val", "test", "test_shift"):
    shutil.copy(f"{src}/{n}.jsonl", f"{dst}/{n}.jsonl")
!ls -la {dst}

## Chọn cấu hình
Đo cục bộ (RTX 4050 6 GB, fp32, batch 1): khoảng 1,25 bản ghi/s; T4 có fp32 cùng cỡ nên giả định 1,0-1,3 bản ghi/s.
Toàn bộ train (27,7k bản ghi) cần khoảng 6-7,5 giờ, quá sát giới hạn phiên 9 giờ khi còn phải chấm điểm,
nên dùng `--max-records 20000` (khoảng 4,5-5,5 giờ). `heads_only` cùng ngân sách bản ghi nhưng nhanh hơn (không lan truyền ngược qua backbone).
Hai tiến trình chạy song song, mỗi cái một GPU. Batch 4 x accum 4 = 16 bản ghi mỗi bước; peak VRAM fp32 batch 1 chỉ khoảng 2,6 GB nên
T4 16 GB dư sức. Có thể chỉnh `BATCH`, `ACCUM`, `MAX_RECORDS` bên dưới (đặt `MAX_RECORDS = 0` để dùng toàn bộ train).
Script dừng nếu RAM máy chủ vượt `--max-rss-gb` (ở đây 24 GB vì Kaggle có khoảng 30 GB).
Lưu ý: chỉ lưu kết quả ở cuối nên nếu phiên bị ngắt giữa chừng sẽ mất phần train đang chạy.

In [ ]:
import os, subprocess, time
REPO = "/kaggle/working/repo"
PI = f"{REPO}/pi"
LOGS = "/kaggle/working/logs"
os.makedirs(LOGS, exist_ok=True)
BATCH, ACCUM, MAX_RECORDS = 4, 4, 20000
common = ["--precision", "fp32", "--epochs", "1", "--batch", str(BATCH), "--accum", str(ACCUM),
          "--max-rss-gb", "24"]
if MAX_RECORDS:
    common += ["--max-records", str(MAX_RECORDS)]

def launch(name, gpu, extra=()):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    cmd = ["python", "scripts/train_jev.py", "--device", "cuda", "--name", name, *common, *extra]
    log = open(f"{LOGS}/{name}.log", "w")
    return subprocess.Popen(cmd, cwd=PI, env=env, stdout=log, stderr=subprocess.STDOUT)

procs = {
    "jev1": launch("jev1", 0),
    "heads_only": launch("heads_only", 1, ["--freeze-backbone"]),
}
print({k: p.pid for k, p in procs.items()})

## Theo dõi (đợi cả hai chạy xong; in 5 dòng cuối của mỗi log mỗi 10 phút)

In [ ]:
def tail(name, n=3):
    with open(f"{LOGS}/{name}.log", errors="replace") as f:
        lines = [l.rstrip() for l in f if "bước" in l or "val loss" in l or "DỪNG" in l or "Traceback" in l]
    return "\n".join(lines[-n:])

while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    for k in procs:
        print(f"[{k}] {tail(k)}\n")
print({k: p.returncode for k, p in procs.items()})
assert all(p.returncode == 0 for p in procs.values()), "một lần train lỗi, xem log trong /kaggle/working/logs"

## Chấm điểm trên test và test_shift (song song hai GPU)

In [ ]:
evals = {}
for gpu, name in enumerate(("jev1", "heads_only")):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    out = f"{PI}/../runs/decider/jev/{name}/eval.log"
    f = open(out, "w")
    evals[name] = subprocess.Popen(
        ["python", "scripts/eval_jev.py", "--name", name, "--device", "cuda", "--precision", "fp32"],
        cwd=PI, env=env, stdout=f, stderr=subprocess.STDOUT)
for name, p in evals.items():
    p.wait()
    print(name, "exit", p.returncode)
for name in evals:
    print("=" * 30, name)
    print(open(f"{REPO}/runs/decider/jev/{name}/eval.log", errors="replace").read()[-6000:])

## Đóng gói kết quả

In [ ]:
import shutil, zipfile
for name in ("jev1", "heads_only"):
    shutil.copy(f"{LOGS}/{name}.log", f"{REPO}/runs/decider/jev/{name}/train.log")
with zipfile.ZipFile("/kaggle/working/jev_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("jev1", "heads_only"):
        base = f"{REPO}/runs/decider/jev/{name}"
        for root, _, files in os.walk(base):
            for fn in files:
                path = os.path.join(root, fn)
                z.write(path, os.path.relpath(path, f"{REPO}/runs/decider/jev"))
    print(z.namelist())
!ls -la /kaggle/working/jev_results.zip